# Neural suite on a Colab GPU

Retrains every neural model in the paper (4 backbones x training regimes x 3 seeds, background-only probes) and
the duplicate-leakage check, then packs the per-image predictions for download. About 2.5 h of training on a T4
(about 3 h including installation and data download); resumable.

1. Runtime -> Change runtime type -> **T4 GPU** (L4 and A100 also work).
2. Optional: set `DRIVE_DIR` below to a Google Drive folder so finished runs survive a disconnect.
3. Runtime -> **Run all**. The last cell downloads `neural_results.zip`; unzip it in the repository root and run
   `bash scripts/reproduce_stats.sh` to rebuild the tables and figures from the new predictions.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/anirudhsengar/waste-background-study'
REPO_REF = None   # a release tag or commit to reproduce, e.g. 'v1.0.0'; None = default branch
PROJ = '/content/waste-background-study'
DRIVE_DIR = None   # e.g. '/content/drive/MyDrive/waste-background-study' to keep results on Drive

out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
assert out.returncode == 0, 'No GPU detected: Runtime -> Change runtime type -> GPU.'
print('GPU:', out.stdout.strip())
if not os.path.exists(PROJ):
    subprocess.run(['git', 'clone', '--depth', '1'] + (['--branch', REPO_REF] if REPO_REF else []) + [REPO_URL, PROJ], check=True)
os.chdir(PROJ)
os.environ['PROJ'] = PROJ
os.environ['DRIVE_DIR'] = DRIVE_DIR or ''
if DRIVE_DIR:
    from google.colab import drive
    drive.mount('/content/drive')

### Environment, data and arrays
Installs the pinned packages, downloads the public datasets (SHA-256 checked) and rebuilds the image arrays.
The `prepare` step must end with **`checksums: ALL MATCH`**.

In [ ]:
%%bash
set -e
cd "$PROJ"
bash scripts/colab_install.sh
SKIP_U2NET=1 bash scripts/download_data.sh
.venv/bin/python -m src.prepare prepare

### Start from empty prediction files
The clone ships the paper's predictions, and `run_gpu.sh` skips every run whose file exists. They are removed
here (`git checkout results` restores them). With `DRIVE_DIR` set, `results/` is kept on Drive instead.

In [ ]:
%%bash
set -e
cd "$PROJ"
if [ -n "$DRIVE_DIR" ]; then
  mkdir -p "$DRIVE_DIR/results/preds" "$DRIVE_DIR/results/logs"
  if [ ! -L results ]; then mv results results_shipped; ln -sfn "$DRIVE_DIR/results" results; fi
else
  rm -f results/preds/*.npz results/logs/*.json results/rq4.json results/rq4.csv results/nn_runs.csv
fi
mkdir -p results/preds results/logs

### Smoke test
Pre-fetches the ImageNet weights and trains each architecture for one epoch on a small subset. Must end with
**`SMOKE TEST OK`**.

In [ ]:
%%bash
set -eo pipefail
cd "$PROJ"
.venv/bin/python -m src.prepare smoke

### Run the neural suite (re-run all cells after a disconnect; finished runs are skipped)

In [ ]:
%%bash
set -eo pipefail
cd "$PROJ"
PYTHON=.venv/bin/python bash scripts/run_gpu.sh 2>&1 | grep --line-buffered -E '===|Traceback|Error|missing'

In [ ]:
import glob, os, subprocess
os.chdir(PROJ)
zip_path = '/content/neural_results.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
files = sorted(glob.glob('results/preds/*.npz') + glob.glob('results/logs/*.json')
               + [f for f in ('results/nn_runs.csv', 'results/rq4.csv', 'results/rq4.json') if os.path.exists(f)])
print(len(glob.glob('results/preds/*.npz')), 'prediction files (96 neural runs expected)')
subprocess.run(['zip', '-q', zip_path] + files, check=True)
print(f'{zip_path}: {os.path.getsize(zip_path) / 1e6:.1f} MB')
try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except Exception as e:
    print('Download it from the Colab file browser:', e)